# Catalog Helpers Notebook

This notebook provides reusable utilities for schema and table management in the CDC pipeline.

## Functions
- `ensure_schema_exists(catalog, schema)` - Create schema if not exists
- `get_existing_schema(spark, table_fqn)` - Get existing Delta table schema
- `create_silver_table_orders(spark, table_fqn)` - Create orders Silver table
- `create_silver_table_products(spark, table_fqn)` - Create products Silver table
- `build_dynamic_merge_set(existing_columns, field_mapping)` - Build MERGE update/insert clauses

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, DecimalType

# Databricks utilities
def ensure_schema_exists(catalog: str, schema: str) -> None:
    """
    Ensure a schema exists in the specified catalog.
    
    Args:
        catalog: Unity Catalog name
        schema: Schema name to create
    """
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    print(f"Schema {catalog}.{schema} is ready")


def get_existing_schema(spark, table_fqn: str) -> StructType | None:
    """
    Get the schema of an existing Delta table.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    
    Returns:
        StructType if table exists, None otherwise
    """
    try:
        df = spark.table(table_fqn)
        return df.schema
    except Exception as e:
        return None


def get_existing_columns(spark, table_fqn: str) -> set:
    """
    Get the set of column names from an existing Delta table.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    
    Returns:
        Set of column names, or empty set if table doesn't exist
    """
    try:
        df = spark.table(table_fqn)
        return set(df.columns)
    except Exception:
        return set()


def create_silver_table_orders(spark, table_fqn: str) -> None:
    """
    Create the Silver orders table with schema for CDC upserts.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    """
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            id INT,
            product_id INT,
            product_name STRING,
            price DECIMAL(12,2),
            created_at TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt TIMESTAMP
        ) USING DELTA
        """
    )
    print(f"Silver orders table {table_fqn} is ready")


def create_silver_table_products(spark, table_fqn: str) -> None:
    """
    Create the Silver products table with schema for CDC upserts.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    """
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            id INT,
            product_name STRING,
            weight DECIMAL(10,2),
            color STRING,
            created_at TIMESTAMP,
            updated_at TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt TIMESTAMP
        ) USING DELTA
        """
    )
    print(f"Silver products table {table_fqn} is ready")


def build_merge_clauses(
    existing_columns: set,
    core_fields: list,
    include_inserted_dt: bool = True,
    include_updated_dt: bool = True
) -> tuple[dict, dict]:
    """
    Build dynamic MERGE update and insert clauses based on existing schema.
    
    Args:
        existing_columns: Set of existing column names in the target table
        core_fields: List of core field names to include in merge
        include_inserted_dt: Whether to include last_inserted_dt field
        include_updated_dt: Whether to include last_updated_dt field
    
    Returns:
        Tuple of (update_set dict, insert_values dict)
    """
    update_set = {}
    insert_values = {}
    
    # Core fields
    for field in core_fields:
        if field in existing_columns and field != "id":
            update_set[field] = f"s.{field}"
        if field in existing_columns:
            insert_values[field] = f"s.{field}"
    
    # Handle timestamp fields
    if include_inserted_dt and "last_inserted_dt" in existing_columns:
        update_set["last_inserted_dt"] = "COALESCE(s.event_time, t.last_inserted_dt)"
        insert_values["last_inserted_dt"] = "s.event_time"
    
    if include_updated_dt and "last_updated_dt" in existing_columns:
        update_set["last_updated_dt"] = "s.event_time"
        insert_values["last_updated_dt"] = "s.event_time"
    
    return update_set, insert_values


def execute_merge(
    spark,
    delta_table: DeltaTable,
    latest_df,
    update_set: dict,
    insert_values: dict,
    join_condition: str = "t.id = s.id",
    delete_condition: str = "s.op = 'd'",
    update_condition: str = "s.op <> 'd'"
) -> None:
    """
    Execute a MERGE operation with dynamic clauses.
    
    Args:
        spark: SparkSession
        delta_table: DeltaTable object for target
        latest_df: DataFrame with latest records to merge
        update_set: Dict of column -> value expressions for UPDATE
        insert_values: Dict of column -> value expressions for INSERT
        join_condition: Join condition string
        delete_condition: Condition for DELETE on matched
        update_condition: Condition for UPDATE on matched
    """
    merge_builder = (
        delta_table.alias("t")
        .merge(latest_df.alias("s"), join_condition)
    )
    
    # Build update clause
    if update_set:
        merge_builder = merge_builder.whenMatchedUpdate(
            condition=update_condition,
            set=update_set
        )
    
    # Build delete clause
    merge_builder = merge_builder.whenMatchedDelete(condition=delete_condition)
    
    # Build insert clause
    if insert_values:
        merge_builder = merge_builder.whenNotMatchedInsert(
            condition=update_condition,
            values=insert_values
        )
    
    merge_builder.execute()


## Usage Example

```python
# Run this helper notebook first to load functions
%run ./helpers/NB_catalog_helpers

# Use the helper functions
ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])
create_silver_table_orders(spark, silver_table_fqn)

# For schema evolution in upsert
existing_columns = get_existing_columns(spark, silver_table_fqn)
update_set, insert_values = build_merge_clauses(existing_columns, core_fields)
execute_merge(spark, delta_table, latest, update_set, insert_values)
```